# MNIST Digit Classification with a CNN

## Goal
Train a convolutional neural network (CNN) to recognize handwritten digits from 0 to 9. Use a custom callback to stop training when accuracy reaches 99.5%.

In [27]:
import tensorflow as tf
import numpy as np


## 1. Load the MNIST Data

MNIST contains 60,000 training images and 10,000 test images.
Each image is a 28 × 28 grayscale picture of a handwritten digit.
The labels are numbers from 0 to 9.

In [28]:
# load dataset 
(train_image,train_label),(test_image,test_label)=tf.keras.datasets.mnist.load_data()

# find shape of the train dataset
print(f"There are {train_image.shape[0]} image examples with shape of ({train_image.shape[1]},{train_image.shape[2]}) ")

There are 60000 image examples with shape of (28,28) 


## 2. Reshape and Normalize the Images

Add a final dimension to each image to represent its single grayscale channel.
This changes each image’s shape from 28 × 28 to 28 × 28 × 1.

Divide the pixel values by 255 to scale them from 0–255 to 0–1.

In [29]:
# Define function
def reshape_normalize(image):
    # reshape image
    image= image.reshape(image.shape[0],image.shape[1],image.shape[2],1)
    # normalize the image
    image=image/255.0
    # return image
    return image


# Call the preprocessing function
train_image=reshape_normalize(train_image)
print(f"Maximum pixel value: {np.max(train_image)} ")
print(f"Shape of training set : {train_image.shape}\nShape of image : {train_image[0].shape} ")

Maximum pixel value: 1.0 
Shape of training set : (60000, 28, 28, 1)
Shape of image : (28, 28, 1) 


## 3. Define the Early-Stopping Callback

At the end of each epoch, the callback checks the training accuracy.
When it reaches 99.5%, the callback stops training and prints a message.

In [30]:
class early_stop(tf.keras.callbacks.Callback):
    def on_epoch_end(self,epochs,log=None):
        if log['accuracy']>=0.995:
            print("Training has reached 99.5% accuracy. Stopping the training")
            self.model.stop_training=True

## 4. Build and Compile the CNN

The model uses a convolution layer to learn image features and a max-pooling layer to reduce their size. `Flatten` prepares the features for the dense layers.

The final layer has 10 units—one for each digit—and uses softmax. The model uses Adam, sparse categorical cross-entropy, and tracks accuracy.

In [31]:
# define model
def convolution_model():
    model=tf.keras.models.Sequential([
        tf.keras.Input(shape=(28,28,1)),
        tf.keras.layers.Conv2D(32,(3,3),activation='relu'),
        tf.keras.layers.MaxPool2D(2,2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128,activation='relu'),
        tf.keras.layers.Dense(10,activation='softmax')
    ])

    model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model


## 5. Train the Model

Train the model for up to 10 epochs. The callback stops training early if accuracy reaches 99.5%.

In [33]:
model=convolution_model()
model.fit(train_image,train_label,epochs=10,callbacks=[early_stop()])

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9567 - loss: 0.1451
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9847 - loss: 0.0496
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9897 - loss: 0.0323
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9935 - loss: 0.0213
Epoch 5/10
1869/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9955 - loss: 0.0142Training has reached 99.5% accuracy. Stopping the training
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9955 - loss: 0.0142


## 6. Evaluate on the Test Data

Evaluate the trained model on test images that were not used during training.
This shows how well the model recognizes unseen handwritten digits.

In [34]:
(_, _), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
test_images = reshape_normalize(test_images)
test_loss, test_accuracy = model.evaluate(test_images, test_labels)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_accuracy:.2%}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9876 - loss: 0.0401
Test loss: 0.0401 | Test accuracy: 98.76%
